In [23]:
import shutil

# Parameters
nvt_time = 2 # ns
npt_time = 5 # ns
dt_eq = 1 # fs
val_sep = 23
cutoff = 1.2 # nm
temperature = "290"
pressure = "1.0"
tau_p = "15.0"

nsteps_nvt = int(nvt_time/2 * 1e6 / dt_eq) # fs * nsteps = ns/2
nsteps_npt = int(npt_time * 1e6 / dt_eq) # same but the npt is just 1 process

# NVT Heating
shutil.copy("templates/nvt_heating.mdp", "./new_nvth.mdp")

# Edit it
with open("./new_nvth.mdp", "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    stripped = line.strip()

    # Title
    if stripped.startswith("title"):
        lines[i] = f"{'title':<{val_sep}} = NVT Heating\n"
    
    # Time
    elif stripped.startswith("nsteps"):
        lines[i] = f"{'nsteps':<{val_sep}} = {nsteps_nvt:<9} ; {nvt_time/2} ns\n"
        lines[i+1] = f"{'dt':<{val_sep}} = {dt_eq/1e3}\n"

    # Cutoffs
    elif stripped.strip().startswith("rcoulomb"):
        lines[i] = f"{'rcoulomb':<{val_sep}} = {cutoff}\n"
    elif stripped.strip().startswith("rvdw"):
        lines[i] = f"{'rvdw':<{val_sep}} = {cutoff}\n"

    # Temperature
    elif stripped.startswith("ref_t"):
        lines[i] = f"{'ref_t':<{val_sep}} = {temperature:<7} {temperature:<7}\n"
    elif stripped.startswith("gen_temp"):
        lines[i] = f"{'gen_temp':<{val_sep}} = {temperature}\n"

with open("./new_nvth.mdp", "w") as f:
    f.writelines(lines)

# NVT Equilibration
shutil.copy("./new_nvth.mdp", "./new_nvte.mdp")
with open("./new_nvte.mdp", "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    stripped = line.strip()

    # Title
    if stripped.startswith("title"):
        lines[i] = f"{'title':<{val_sep}} = NVT Equilibration\n"
    
    # Time - Same as Heating
    # Cutoffs - Same as Heating
    # Temperature - Same as Heating

    elif stripped.startswith("gen_vel"):
        lines[i] = f"{'gen_vel':<{val_sep}} = no\n"
        del lines[i+1:i+3]

with open("./new_nvte.mdp", "w") as f:
    f.writelines(lines)

# NPT
shutil.copy("./new_nvte.mdp", "./new_npt.mdp")
with open("./new_npt.mdp", "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    stripped = line.strip()

    # Title
    if stripped.startswith("title"):
        lines[i] = f"{'title':<{val_sep}} = NPT\n"
    
    # Time
    elif stripped.startswith("nsteps"):
        lines[i] = f"{'nsteps':<{val_sep}} = {nsteps_npt:<9} ; {npt_time} ns\n"
        lines[i+1] = f"{'dt':<{val_sep}} = {dt_eq/1e3}\n"

    # Cutoffs - Same as Heating
    # Temperature - Same as Heating

    # Pressure Coupling
    elif stripped.startswith("pcoupl"):
        pressure_block = [
            f"{'pcoupl':<{val_sep}} = C-rescale\n",
            f"{'pcoupltype':<{val_sep}} = isotropic\n",
            f"{'tau_p':<{val_sep}} = {tau_p}\n",
            f"{'ref_p':<{val_sep}} = {pressure}\n",
            f"{'compressibility':<{val_sep}} = 4.5e-5\n",
            f"{'refcoord_scaling':<{val_sep}} = com\n"
        ]
        lines[i:i+1] = pressure_block
        break

with open("./new_npt.mdp", "w") as f:
    f.writelines(lines)

# MD Run
dt_md = 2
md_time = 25
nsteps_md = int(md_time * 1e6 / dt_md)
nstxout_set = False

shutil.copy("./new_npt.mdp", "./new_md.mdp")
with open("./new_md.mdp", "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    stripped = line.strip()

    # Title
    if stripped.startswith("title"):
        lines[i] = f"{'title':<{val_sep}} = MD Run\n"

    # Remove position restrain
    elif stripped.startswith("define"):
        del lines[i]
    
    # Time
    elif stripped.startswith("nsteps"):
        lines[i] = f"{'nsteps':<{val_sep}} = {nsteps_md:<9} ; {md_time} ns\n"
        lines[i+1] = f"{'dt':<{val_sep}} = {dt_md/1e3}\n"

        # Insert COM Restrains
        lines.insert(i + 2, f"{'comm-mode':<{val_sep}} = Angular\n")
        lines.insert(i + 3, f"{'comm-grps':<{val_sep}} = Protein\n")
    
    # Out Kinematics
    elif stripped.startswith("nstxout") and (nstxout_set == False):
        lines[i] = f"{'nstxout':<{val_sep}} = 0\n"
        lines[i + 1] = f"{'nstvout':<{val_sep}} = 0\n"
        lines.insert(i + 2, f"{'nstfout':<{val_sep}} = 0\n")
        nstxout_set = True

    # Output Macros
    elif stripped.startswith("nstenergy"):
        lines[i + 1] = f"{'nstlog':<{val_sep}} = 50000\n"
        lines.insert(i + 2, f"{'nstxout-compressed':<{val_sep}} = 50000\n")
        lines.insert(i + 3, f"{'compressed-x-grps':<{val_sep}} = System\n")

    # The other stuff remains the same

with open("./new_md.mdp", "w") as f:
    f.writelines(lines)
